In [1]:
!pip -q install pymorphy3

In [2]:
!pip -q install gensim==4.3.2

In [3]:
!pip -q install scipy==1.12.0

In [4]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

import warnings
warnings.filterwarnings('ignore')


%matplotlib inline

In [5]:
df = pd.read_csv('news.csv')
df.sort_values(by = ['event'], inplace = True)
df.head()

,Unnamed: 0,event,text
297,297,CNN показала фильм «Владимир Путин — самый вли...,CNN: ВЛАДИМИР ПУТИН - САМЫЙ МОГУЩЕСТВЕННЫЙ ЧЕ...
298,298,CNN показала фильм «Владимир Путин — самый вли...,"РБК (rbc.ru), Москва, 14 марта 2017 КРЕМЛЬ ПР..."
678,678,SpaceX впервые в истории запустила и посадила ...,"Newstes.ru, Москва, 31 марта 2017 ПОЛЕТ НОРМА..."
679,679,SpaceX впервые в истории запустила и посадила ...,"Vistanews.ru, Москва, 31 марта 2017 SPACE X В..."
680,680,SpaceX впервые в истории запустила и посадила ...,"Russia Today (russian.rt.com), Москва, 31 мар..."


In [6]:
df.event.value_counts()

,count
event,
Чемпионат мира по хоккею,102
теракт произошел в центре Лондона,100
Кубок конфедерации FiFA,100
Саммит G20,100
Путин и Меркель в Сочи,100
Умер Евгений Евтушенко,100
Правительство внесло в Госдуму законопроект о курортных сборах,100
Ураган в Москве,100
Теракт в Барселоне,100


### Препроцессинг

In [7]:
import re
from nltk.corpus import stopwords

In [8]:
from pymystem3 import Mystem
m = Mystem()

In [9]:
import pymorphy3
m = pymorphy3.MorphAnalyzer()

In [10]:
import nltk
nltk.download('stopwords')

[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


True

In [11]:
mystopwords = stopwords.words('russian') + [
    'это', 'наш' , 'тыс', 'млн', 'млрд', 'также',  'т', 'д',
    'который','прошлый','сей', 'свой', 'наш', 'мочь', 'такой'
]
ru_words = re.compile("[А-Яа-я]+")


def words_only(text):
    return " ".join(ru_words.findall(text))


# def lemmatize(text, mystem=m):
#     try:
#         return "".join(m.lemmatize(text)).strip()
#     except:
#         return " "

def lemmatize(text, mystem=m):
    try:
        return " ".join([m.parse(w)[0].normal_form for w in text.split(' ')]).strip()
    except:
        return " "

def remove_stopwords(text, mystopwords = mystopwords):
    try:
        return " ".join([token for token in text.split() if not token in mystopwords])
    except:
        return ""


def preprocess(text):
    return remove_stopwords(lemmatize(words_only(text.lower())))

In [12]:
m.parse("замок")

[Parse(word='замок', tag=OpencorporaTag('NOUN,inan,masc sing,nomn'), normal_form='замок', score=0.428571, methods_stack=((DictionaryAnalyzer(), 'замок', 141, 0),)),
 Parse(word='замок', tag=OpencorporaTag('NOUN,inan,masc sing,accs'), normal_form='замок', score=0.428571, methods_stack=((DictionaryAnalyzer(), 'замок', 141, 3),)),
 Parse(word='замок', tag=OpencorporaTag('VERB,perf,intr masc,sing,past,indc'), normal_form='замокнуть', score=0.142857, methods_stack=((DictionaryAnalyzer(), 'замок', 764, 1),))]

In [ ]:
%%time
df.text = df.text.apply(preprocess)

## Генсим

### TfidfModel

In [ ]:
from gensim.corpora import Dictionary
from gensim.models import TfidfModel
from gensim import similarities

In [ ]:
texts = [text.split() for text in df.text]
dictionary = Dictionary(texts)
corpus = [dictionary.doc2bow(text) for text in texts]

In [ ]:
dictionary.doc2bow(['спорт', 'футбол', 'мяч'])

In [ ]:
# поизучаем, что здесь происходит
# corpus

Косинусная близость:


$$similarity=\cos(\theta )={\mathbf {A} \cdot \mathbf {B}  \over \|\mathbf {A} \|\|\mathbf {B} \|}={\frac {\sum \limits _{i=1}^{n}{A_{i}B_{i}}}{{\sqrt {\sum \limits _{i=1}^{n}{A_{i}^{2}}}}{\sqrt {\sum \limits _{i=1}^{n}{B_{i}^{2}}}}}}$$

Вычисление сходства по косинусной мере на векторах $tf-idf$:

In [ ]:
tfidf = TfidfModel(corpus)
corpus_tfidf = tfidf[corpus]

In [ ]:
corpus_tfidf

In [ ]:
# %%time
index = similarities.MatrixSimilarity(corpus_tfidf)
sims = index[corpus_tfidf]

In [ ]:
len(corpus)

In [ ]:
sims[0]

In [ ]:
sims.shape

In [ ]:
import seaborn as sns

In [ ]:
print(np.median(sims))
sims  = (sims - sims.min())/(sims.max() - sims.min())
plt.figure(figsize = (10,10))
sns.heatmap(data=sims, cmap = 'Spectral').set(xticklabels=[],yticklabels=[])
plt.title("Матрица близости")
plt.show()

### Латентно-семантический анализ (LSA, LSI)


Сингулярное разложение: $M'_k = U \Sigma_k V^t_k$

* $M$ – матрица слово-документ
* $U$ – матрица документ-тема
* $k$ – число тем
* $V$ - матрица слово-тема

In [ ]:
from gensim.models import lsimodel

In [ ]:
%%time
lsi = lsimodel.LsiModel(corpus=corpus_tfidf, id2word=dictionary, num_topics=30)

In [ ]:
lsi.show_topics(5)

Как снижение размерности влияет на матрицу близости:

In [ ]:
corpus_lsi = lsi[corpus]
index = similarities.MatrixSimilarity(lsi[corpus])
sims = index[corpus_lsi]
sims  = (sims - sims.min())/(sims.max() - sims.min())
print(np.median(sims))
plt.figure(figsize = (10,10))
sns.heatmap(data=sims, cmap = 'Spectral').set(xticklabels=[], yticklabels=[])
plt.title("Матрица близости")
plt.show()

### Главные компоненты

In [ ]:
lsi[corpus[0]]

In [ ]:
X = [0] * len(df)
Y = [0] * len(df)
for i in range(len(df)):
    vec = corpus[i]
    LSI_topics = (lsi[vec])
    for topic in LSI_topics:
        if topic[0] == 0:
            X[i] = topic[1]
        elif topic[0] == 1:
            Y[i] = topic[1]

vis_df = pd.DataFrame({'X': X, 'Y': Y, 'topic' : df.event})

In [ ]:
sns.FacetGrid(vis_df, hue="topic", height = 10).map(plt.scatter, "X", "Y").add_legend()

### Truncated SVD и TSNE

#### t-SNE
t-SNE (t-distributed Stochastic Neighbor Embedding) — техника нелинейного снижения размерности и визуализации многомерных переменных. Она разработана специально для данных высокой размерности Л. ван дер Маатеном и Д. Хинтоном.

t-SNE — это итеративный алгоритм, основанный на вычислении попарных расстояний между всеми объектами (в том числе поэтому он довольно медленный).

In [ ]:
from sklearn.manifold import TSNE
from sklearn.decomposition import TruncatedSVD
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import Normalizer
from sklearn.feature_extraction.text import TfidfVectorizer

In [ ]:
vectors = TfidfVectorizer().fit_transform(df.text) # строим тф-идф матрицу документ-слово
X_reduced = TruncatedSVD(n_components=5, random_state=0).fit_transform(vectors) # оставляем n главных компонент

In [ ]:
X_reduced

In [ ]:
# %%time
X_embedded = TSNE(n_components=2, perplexity=5, verbose=0).fit_transform(X_reduced) # строим мэппинг в двухмерную плоскость с TSNE

In [ ]:
vis_df = pd.DataFrame({'X': X_embedded[:, 0], 'Y': X_embedded[:, 1], 'topic' : df.event})
sns.FacetGrid(vis_df, hue="topic", height=10).map(plt.scatter, "X", "Y").add_legend()